# 🌊 WaveWash & 🛡️ SplitSafe: Audio ML Data Quality Quickstart
### End-to-End Audio Quality Control, Non-Destructive De-clipping & Pre-Training Split Leakage Prevention

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BoobalanArjunan/wavewash/blob/main/notebooks/quickstart.ipynb)
[![PyPI - WaveWash](https://img.shields.io/pypi/v/wavewash)](https://pypi.org/project/wavewash/)
[![PyPI - SplitSafe](https://img.shields.io/pypi/v/splitsafe)](https://pypi.org/project/splitsafe/)
[![License: MIT](https://img.shields.io/badge/License-MIT-blue.svg)](https://opensource.org/licenses/MIT)

Machine Learning models (Whisper, Voice Cloning, Generative Music, TTS) are highly vulnerable to **corrupted audio signals** (digital clipping, DC bias, noise) and **benchmark data leakage** (the same recording appearing across train and test sets).

This interactive tutorial demonstrates how to:
1. **Audit & Repair Audio** with `wavewash` (Non-destructive spline peak reconstruction +71.8% MSE reduction).
2. **Prevent Train/Test Leakage** with `splitsafe` (Acoustic fingerprint constellation matching across splits).
3. **Integrate into PyTorch** for zero-friction production ML pipelines.

## 1. Installation
Install `wavewash` and `splitsafe` directly from PyPI into your Google Colab environment:

In [ ]:
# Install packages directly from PyPI
!pip install -q wavewash splitsafe soundfile matplotlib numpy

## 2. Part 1: Signal-Level Quality Control with WaveWash
Let's synthesize an audio clip with common real-world defects:
* **Severe Digital Clipping** (clamped at 0 dBFS causing flat-topped distortion)
* **DC Offset Bias** (signal centered off-zero, stealing dynamic headroom)

Listen to the audio player below to hear the harsh buzzy distortion.

In [ ]:
import numpy as np
import soundfile as sf
import IPython.display as ipd
import matplotlib.pyplot as plt
from pathlib import Path

import wavewash as ww

# Setup a demo workspace
work_dir = Path("./audio_demo")
work_dir.mkdir(exist_ok=True)

# Synthesize 3-second harmonic speech tone (Fundamental 220Hz + harmonics)
sr = 16000
t = np.linspace(0, 3.0, int(sr * 3.0), endpoint=False)
clean_signal = (
    0.60 * np.sin(2 * np.pi * 220 * t) +
    0.25 * np.sin(2 * np.pi * 440 * t) +
    0.15 * np.sin(2 * np.pi * 880 * t)
)

# Add DC offset (+0.08) and simulate extreme microphone preamp clipping
corrupted = clean_signal + 0.08
clipped_signal = np.clip(corrupted * 1.9, -0.98, 0.98)

dirty_path = work_dir / "dirty_speech_sample.wav"
sf.write(dirty_path, clipped_signal, sr)

print(f"Created test file: {dirty_path}")
print("Listen below: Notice the harsh buzzy distortion caused by clipped peaks and DC shift:")
ipd.Audio(dirty_path)

## 3. Automated Diagnosis with WaveWash
Let's run `wavewash.analyze_file()` against the official `'asr'` (Speech Recognition) profile.

In [ ]:
# Analyze file with pre-calibrated Speech Recognition profile
profile = ww.PROFILES["asr"]
report = ww.analyze_file(dirty_path, profile)

print(f"Verdict: {'✅ PASS' if report.passed else '❌ FAIL'}")
print("Detected Issues:")
for issue in report.issues:
    print(f"  - [{issue.code}] {issue.message} (Severity: {issue.severity})")

## 4. Deterministic Non-Destructive Repair
WaveWash features a non-destructive bounded cubic spline de-clipping engine (+71.8% MSE reduction) with proportional ceiling attenuation, ensuring no clipping overshoot occurs while removing DC bias without altering unaffected portions.

In [ ]:
# Load audio and execute deterministic repair pipeline
audio, sample_rate = ww.load_audio(dirty_path)
repaired_audio, repaired_sr, fixes = ww.apply_fix_pipeline(
    audio, sample_rate, profile, report.issues
)

clean_path = work_dir / "repaired_speech_sample.wav"
ww.save_audio(repaired_audio, clean_path, repaired_sr)

print(f"Successfully Applied Fixes: {fixes}")
print("Listen below: The buzz is eliminated and smooth acoustic harmonics are restored!")
ipd.Audio(clean_path)

## 5. Visual Peak Reconstruction Waveform Plot
Compare the zoomed-in waveform before and after WaveWash repair:

In [ ]:
# Zoom into a 25ms window to inspect individual audio peaks
zoom_start = int(sr * 1.000)
zoom_end = int(sr * 1.025)

plt.figure(figsize=(12, 4), dpi=120)
plt.plot(
    t[zoom_start:zoom_end],
    clipped_signal[zoom_start:zoom_end],
    label="Clipped Audio (Flat-topped distortion)",
    color="#ef4444",
    linewidth=2,
    linestyle="--",
)
plt.plot(
    t[zoom_start:zoom_end],
    repaired_audio[zoom_start:zoom_end],
    label="WaveWash Restored (Smooth cubic spline reconstruction)",
    color="#06b6d4",
    linewidth=2,
)

plt.title("WaveWash Peak Reconstruction & De-Clipping", fontsize=13, fontweight="bold")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.grid(True, alpha=0.3)
plt.legend(loc="upper right", frameon=True)
plt.tight_layout()
plt.show()

---
## 6. Part 2: Pre-Training Dataset Leakage Check with SplitSafe
Now that individual files are clean, we must prevent **cross-split contamination**.
If the same speech recording (even if re-encoded to FLAC/MP3, resampled, trimmed, or gain-shifted) appears in both `train/` and `test/`, your model evaluation is compromised.

Let's construct a dataset where a training clip was trimmed and placed in the test split:

In [ ]:
# Create mock train/ and test/ folders
train_dir = work_dir / "dataset" / "train"
test_dir = work_dir / "dataset" / "test"
train_dir.mkdir(parents=True, exist_ok=True)
test_dir.mkdir(parents=True, exist_ok=True)

# Synthesize unique recordings
t_clip = np.linspace(0, 2.0, int(sr * 2.0), endpoint=False)
clip_train_1 = np.sin(2 * np.pi * 320 * t_clip)
clip_train_2 = np.sin(2 * np.pi * 480 * t_clip)
clip_test_clean = np.sin(2 * np.pi * 750 * t_clip)

sf.write(train_dir / "spk1_recording_a.wav", clip_train_1, sr)
sf.write(train_dir / "spk2_recording_b.wav", clip_train_2, sr)
sf.write(test_dir / "spk3_test_unique.wav", clip_test_clean, sr)

# SIMULATE LEAKAGE: Trim 0.3s off train recording A and copy to test folder
leaked_clip = clip_train_1[int(sr * 0.3):int(sr * 1.7)] * 0.85
sf.write(test_dir / "spk1_leaked_sample.wav", leaked_clip, sr)

print("Dataset created with hidden cross-split leak.")

## 7. Run SplitSafe Leakage Auditor
`splitsafe` uses acoustic landmark constellation hashes to find matching recordings across folders instantly, even across edits:

In [ ]:
# Run splitsafe audit via CLI
!splitsafe check ./audio_demo/dataset/train ./audio_demo/dataset/test

---
## 8. PyTorch DataLoader Drop-In Integration
You can plug WaveWash directly into your PyTorch training pipeline to clean data on-the-fly without altering original disk files:

```python
import torch
from torch.utils.data import Dataset
from pathlib import Path
import wavewash as ww

class CleanSpeechDataset(Dataset):
    def __init__(self, file_paths, profile_name="asr", auto_fix=True):
        self.file_paths = [Path(p) for p in file_paths]
        self.profile = ww.PROFILES[profile_name]
        self.auto_fix = auto_fix

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        report = ww.analyze_file(path, self.profile)
        audio, sr = ww.load_audio(path)
        
        # Deterministic repair on-the-fly
        if not report.passed and self.auto_fix:
            audio, sr, _ = ww.apply_fix_pipeline(audio, sr, self.profile, report.issues)
            
        return torch.from_numpy(audio).float(), sr, path.name
```

🎉 **Summary:** Your training data is now mathematically validated, free from clipping distortion, and guaranteed leak-free for production ML.